# 2.3 Integrating external databases


# 2.3.1 Example 1: Uniprot integration

Think of a scenario that a biologist wants to learn about the protein **p53** (a critical tumor suppressor) -- specifically its function, domains, and disease associations, without manually reading through dense database entries.

In this example we will be using Anthropic's `claude-opus-4-5` model. So you will have to set an Anthropic API Key before starting. 

## Step 1: Query the UniProt REST API
UniProt offers a free, no-authentication REST API. You can fetch data for p53 (human) using its UniProt accession ID `P04637`:

In [7]:
# Install the necessary packages
!pip install requests anthropic py3Dmol python-dotenv


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import requests

# Fetch p53 human protein entry from UniProt
accession = "P04637"
url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"

response = requests.get(url)
data = response.json()

# Extract key fields
protein_name = data['proteinDescription']['recommendedName']['fullName']['value']
organism = data['organism']['scientificName']
function_text = next(
    c['texts'][0]['value']
    for c in data['comments']
    if c['commentType'] == 'FUNCTION'
)
sequence = data['sequence']['value']

print(f"Protein: {protein_name}")
print(f"Organism: {organism}")
print(f"Function: {function_text[:300]}...")
print(f"Sequence length: {data['sequence']['length']} aa")

Protein: Cellular tumor antigen p53
Organism: Homo sapiens
Function: Multifunctional transcription factor that induces cell cycle arrest, DNA repair or apoptosis upon binding to its target DNA sequence (PubMed:11025664, PubMed:12524540, PubMed:12810724, PubMed:15186775, PubMed:15340061, PubMed:17317671, PubMed:17349958, PubMed:19556538, PubMed:20673990, PubMed:209594...
Sequence length: 393 aa


## Step 2: Pass the Data to an LLM
Now feed the retrieved data into an LLM (e.g., Claude via the API) with a biologically meaningful prompt. 


> [!warning]
> Make sure you have set an `ANTHROPIC_API_KEY` key. Add as a Google Colab Secret or as an environment variable.



In [2]:
import os
from dotenv import load_dotenv

def get_api_key():
    """Load Anthropic API key from Colab secrets or environment variable."""
    try:
        from google.colab import userdata
        return userdata.get('ANTHROPIC_API_KEY')
    except ImportError:
        # Not in Colab — fall back to environment variable
        load_dotenv() 
        api_key = os.environ.get('ANTHROPIC_API_KEY')
        if not api_key:
            raise ValueError(
                "API key not found. Please set the ANTHROPIC_API_KEY environment variable.\n"
                "You can do this by running the following in your terminal:\n"
                "  export ANTHROPIC_API_KEY='your-key-here'\n"
                "Or add it to a .env file in your project root."
            )
        return api_key


In [5]:
import anthropic

client = anthropic.Anthropic(api_key=get_api_key())  # uses ANTHROPIC_API_KEY env variable

# Build a prompt using the retrieved UniProt data
prompt = f"""
I retrieved the following information about a protein from UniProt:

Protein name: {protein_name}
Organism: {organism}
Function annotation: {function_text}
Sequence length: {data['sequence']['length']} amino acids

Please do the following:
1. Summarize what this protein does in plain language suitable for an undergraduate student.
2. Explain why this protein is clinically important.
3. Suggest 2–3 follow-up questions a researcher might want to investigate.
"""

message = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    messages=[{"role": "user", "content": prompt}]
)

print(message.content[0].text)

# Summary of Cellular Tumor Antigen p53 (TP53)

## 1. What Does This Protein Do? (Plain Language Summary)

**p53 is often called the "guardian of the genome"** because its main job is to protect your cells from becoming cancerous.

Think of p53 as a quality control manager in your cells. When it detects problems—like DNA damage or stress—it can do three things:

1. **Stop cell division** (hit the brakes so the cell can fix the damage)
2. **Activate DNA repair** (call in the repair crew)
3. **Trigger cell death (apoptosis)** (if the damage is too severe, it tells the cell to self-destruct rather than become dangerous)

p53 works as a **transcription factor**, meaning it turns specific genes on or off by binding to DNA. For example, it activates genes that stop the cell cycle and genes that promote cell death when necessary. It also interacts with other proteins that can either enhance or inhibit its protective functions.

Additionally, p53 has some bonus jobs: it helps regulate the body

## Step 3: Go Deeper — Ask About Disease Variants
You can pull disease variant annotations from the same UniProt entry and ask the LLM to interpret them:

In [6]:
# Extract disease associations
diseases = [
    c for c in data['comments']
    if c['commentType'] == 'DISEASE'
]

disease_summary = "\n".join([
    d.get('disease', {}).get('description', '') for d in diseases
])

variant_prompt = f"""
The following diseases are associated with mutations in {protein_name}:

{disease_summary}

What do these associations tell us about the protein's role in cancer biology? 
What types of mutations (gain-of-function vs loss-of-function) are typically seen?
"""

response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    messages=[{"role": "user", "content": variant_prompt}]
)

print(response.content[0].text)

# TP53 (Cellular Tumor Antigen p53) in Cancer Biology

## What These Associations Reveal About p53's Role

### 1. **Guardian of the Genome Function**
The extraordinary diversity of cancer types associated with p53 mutations demonstrates that p53 is a **universal tumor suppressor** - not tissue-specific, but essential for preventing malignancy across virtually all cell types:
- Epithelial tissues (esophageal, lung, skin cancers)
- Mesenchymal tissues (sarcomas)
- Neural tissues (brain tumors, choroid plexus papilloma)
- Hematopoietic tissues (leukemia, bone marrow failure)
- Endocrine tissues (adrenocortical carcinoma)

### 2. **Critical Cellular Functions Protected by p53**

| Function | Evidence from Disease Associations |
|----------|-----------------------------------|
| **Cell cycle arrest** | Rapidly dividing tissues (skin, GI tract) develop cancers |
| **DNA damage response** | UV-exposed skin cancers (BCC, SCC) |
| **Apoptosis induction** | Childhood cancers suggest failed elimi

# 2.3.2 Example 2: PDB databank integration

Now let's look at a scenario where a biologist wants to understand the 3D structure of p53 bound to DNA — what the structure looks like, what experimental method was used, and what the structural features mean biologically.

## Step 1: Search PDB for p53 Structures

In [7]:
import requests

# Search PDB for human p53 structures
search_url = "https://search.rcsb.org/rcsbsearch/v2/query"

query = {
    "query": {
        "type": "terminal",
        "service": "full_text",
        "parameters": {"value": "p53 DNA binding human"}
    },
    "return_type": "entry",
    "request_options": {
        "paginate": {"start": 0, "rows": 5}
    }
}

response = requests.post(search_url, json=query)
results = response.json()

pdb_ids = [r['identifier'] for r in results['result_set']]
print("Found PDB entries:", pdb_ids)


Found PDB entries: ['1T4W', '6TOQ', '3FE7', '3FEA', '1DXS']


## Step 2: Fetch Detailed Metadata for a Structure

In [9]:
# Use a well-known p53-DNA complex structure
pdb_id = "2OCJ"

# Fetch entry-level metadata
entry_url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
entry_data = requests.get(entry_url).json()

# Fetch polymer entity info (the protein chains)
entity_url = f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/1"
entity_data = requests.get(entity_url).json()

# Pull out key fields
title = entry_data['struct']['title']
method = entry_data['exptl'][0]['method']

# Resolution: try multiple possible field names, then fall back to rcsb_entry_info
def get_resolution(entry_data):
    # Try refine block first (X-ray)
    refine = entry_data.get('refine', [{}])
    for field in ['ls_d_res_high', 'ls_d_res_low', 'resolution_high']:
        val = refine[0].get(field)
        if val is not None:
            return val
    # Fall back to rcsb_entry_info (works for cryo-EM and other methods too)
    return entry_data.get('rcsb_entry_info', {}).get('resolution_combined', [None])[0]

resolution = get_resolution(entry_data)

organism = entity_data['rcsb_entity_source_organism'][0]['scientific_name']
description = entity_data['rcsb_polymer_entity']['pdbx_description']

print(f"Title: {title}")
print(f"Method: {method}")
print(f"Resolution: {resolution} Å" if resolution else "Resolution: N/A")
print(f"Organism: {organism}")
print(f"Entity: {description}")

Title: Human p53 core domain in the absence of DNA
Method: X-RAY DIFFRACTION
Resolution: 2.05 Å
Organism: Homo sapiens
Entity: P53 TUMOR SUPPRESSOR


## Step 3: Fetch the Protein Sequence from PDB

In [10]:
# Get the sequence of the p53 chain in this structure
sequence_url = f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/1"
seq_data = requests.get(sequence_url).json()

sequence = seq_data['entity_poly']['pdbx_seq_one_letter_code_can']
seq_length = len(sequence)

print(f"Sequence length in structure: {seq_length} aa")
print(f"First 60 aa: {sequence[:60]}")

Sequence length in structure: 219 aa
First 60 aa: SSSVPSQKTYQGSYGFRLGFLHSGTAKSVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPP


In [11]:
# Visualize the p53-DNA Structure

import py3Dmol

# Create a viewer and load the structure directly from RCSB
view = py3Dmol.view(query='pdb:2OCJ', width=800, height=500)

# Style the different components
view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'lightblue'}})   # p53 chain A
view.setStyle({'chain': 'B'}, {'cartoon': {'color': 'lightgreen'}})  # p53 chain B
view.setStyle({'chain': 'C'}, {'cartoon': {'color': 'lightyellow'}}) # DNA chain
view.setStyle({'chain': 'D'}, {'cartoon': {'color': 'lightyellow'}}) # DNA chain

# Highlight DNA as sticks so it stands out
view.addStyle({'resn': ['DA', 'DT', 'DG', 'DC']}, {'stick': {'colorscheme': 'orangeCarbon'}})

# Zoom to fit
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Step 4: Ask the LLM to Interpret the Structure

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=get_api_key())

structure_prompt = f"""
I retrieved the following information about a protein crystal structure from the RCSB PDB:

PDB ID: {pdb_id}
Title: {title}
Experimental method: {method}
Resolution: {resolution} Å
Organism: {organism}
Protein: {description}
Sequence length in structure: {seq_length} amino acids

Please do the following:
1. Explain what this structure represents in plain language.
2. Explain what X-ray crystallography is and what resolution means — 
   is {resolution} Å considered good, and what level of detail does it provide?
3. Why is studying p53 in complex with DNA particularly important 
   for understanding its tumor suppressor function?
4. What are the limitations of studying a protein this way 
   (i.e., as a static crystal structure)?
"""

message = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    messages=[{"role": "user", "content": structure_prompt}]
)

print(message.content[0].text)

## Step 5: Compare Multiple Structures
A powerful follow-up — fetch several p53 structures and ask the LLM to reason about the differences:

In [ ]:
# Fetch metadata for several p53 structures
structures_info = []
for pid in pdb_ids[:3]:  # use first 3 results from search
    e = requests.get(f"https://data.rcsb.org/rest/v1/core/entry/{pid}").json()
    try:
        structures_info.append({
            "pdb_id": pid,
            "title": e['struct']['title'],
            "method": e['exptl'][0]['method'],
            "resolution": e.get('refine', [{}])[0].get('ls_d_res_high', 'N/A')
        })
    except KeyError:
        continue

comparison_prompt = f"""
Here are several PDB structures related to p53:

{structures_info}

1. What differences between these structures might be biologically meaningful?
2. Why might researchers solve multiple structures of the same protein?
3. How might differences in resolution affect which structure a researcher chooses for drug design?
"""

response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    messages=[{"role": "user", "content": comparison_prompt}]
)

print(response.content[0].text)

## Summary of PDB API Endpoints Used
| Endpoint  | What It Returns |
|--|--|
| `search.rcsb.org/rcsbsearch/v2/query` | Search for structures by keyword |
| `data.rcsb.org/rest/v1/core/entry/{id}` | Title, method, resolution, deposition date | 
| `data.rcsb.org/rest/v1/core/polymer_entity/{id}/1` | Chain info, sequence, organism|